# Head Cooperation Analysis

How do attention heads cooperate and compete? Analyze within-layer cooperation,
cross-layer alignment, redundancy, interference, and specialization diversity.

## Why This Matters

Attention head cooperation reveals how heads route information between positions. Understanding attention mechanics is central to reverse-engineering transformer circuits, as attention patterns determine what information each component can access and transform.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.head_cooperation_analysis import (
    within_layer_cooperation, cross_layer_head_alignment,
    head_redundancy_analysis, head_output_interference,
    head_specialization_diversity,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Within-Layer Cooperation

Do heads in the same layer pull in the same direction?

In [ ]:
result = within_layer_cooperation(model, tokens, layer=0, position=-1)
coop = 'cooperative' if result['is_cooperative'] else 'competitive'
print(f"Layer {result['layer']}, position {result['position']}: {coop}")
print(f"Cooperative: {result['n_cooperative_pairs']}, Competing: {result['n_competing_pairs']}\n")
for p in result['pairs']:
    tag = '+' if p['cooperates'] else '-'
    print(f"  H{p['head_a']}↔H{p['head_b']}: cos={p['cosine_similarity']:+.4f} [{tag}]")

## Cross-Layer Head Alignment

Does head 0 do similar things at different depths?

In [ ]:
result = cross_layer_head_alignment(model, tokens, head=0)
cons = 'CONSISTENT' if result['is_consistent'] else 'variable'
print(f"Head {result['head']}: mean alignment = {result['mean_alignment']:.4f} [{cons}]\n")
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: norm={p['output_norm']:.4f}, "
          f"align_prev={p['alignment_to_previous']:+.4f}")

## Head Redundancy

Are any heads producing similar outputs?

In [ ]:
result = head_redundancy_analysis(model, tokens, layer=0)
print(f"Redundant heads: {result['n_redundant']}/{len(result['per_head'])}\n")
for h in result['per_head']:
    tag = 'REDUNDANT' if h['is_redundant'] else 'unique'
    print(f"  Head {h['head']}: norm={h['output_norm']:.4f}, "
          f"max_sim={h['max_similarity_to_other']:.4f} [{tag}]")
if result['redundant_pairs']:
    print('\nRedundant pairs:')
    for p in result['redundant_pairs']:
        print(f"  H{p['head_a']}↔H{p['head_b']}: sim={p['similarity']:.4f}")

## Head Output Interference

Do head outputs cancel each other out?

In [ ]:
result = head_output_interference(model, tokens, layer=0)
interf = 'YES' if result['has_significant_interference'] else 'no'
print(f"Significant interference: {interf}")
print(f"Mean ratio: {result['mean_interference_ratio']:.4f}\n")
for p in result['per_position']:
    tag = 'CANCEL' if p['has_interference'] else 'ok'
    print(f"  Pos {p['position']}: sum_norms={p['sum_of_norms']:.4f}, "
          f"norm_sum={p['norm_of_sum']:.4f}, ratio={p['interference_ratio']:.4f} [{tag}]")

## Specialization Diversity

How diverse are the attention patterns?

In [ ]:
result = head_specialization_diversity(model, tokens, layer=0)
print(f"Diversity: {result['mean_diversity']:.4f}")
print(f"Unique heads: {result['n_unique_heads']}/{len(result['per_head'])}\n")
for h in result['per_head']:
    tag = 'UNIQUE' if h['is_unique'] else 'similar'
    print(f"  Head {h['head']}: sim_to_others={h['mean_similarity_to_others']:.4f}, "
          f"entropy={h['pattern_entropy']:.4f} [{tag}]")